# 01 - Smart-Meter Exploratory Data Analysis (EDA)

**Project:** Trustworthy AMI Forecasting Under Behavioral Distribution Shift (`shift-ami`)  
**Objective:** Inspect smart-meter consumption profiles, missingness, cohort aggregations, and data quality distributions.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from shift_ami.config import load_config

# Load configuration
config = load_config("../configs/smoke.yaml")
print(f"Config loaded. Raw directory: {config.paths.raw_dir}")

## 1. Load Processed Cohort Aggregates

In [ ]:
df_total = pd.read_parquet(config.paths.processed_dir / "cohort_total_halfhourly.parquet")
df_dtou = pd.read_parquet(config.paths.processed_dir / "cohort_dtou_halfhourly.parquet")
df_std = pd.read_parquet(config.paths.processed_dir / "cohort_standard_halfhourly.parquet")

print(f"Total Cohort: {len(df_total):,} rows | {df_total['timestamp'].min()} to {df_total['timestamp'].max()}")
print(f"dToU Cohort: {len(df_dtou):,} rows")
print(f"Standard Cohort: {len(df_std):,} rows")
df_total.head()

## 2. Diurnal Load Profile by Cohort

In [ ]:
df_total['hour'] = df_total['timestamp'].dt.hour
df_dtou['hour'] = df_dtou['timestamp'].dt.hour
df_std['hour'] = df_std['timestamp'].dt.hour

hourly_total = df_total.groupby('hour')['energy_kwh'].mean()
hourly_dtou = df_dtou.groupby('hour')['energy_kwh'].mean()
hourly_std = df_std.groupby('hour')['energy_kwh'].mean()

plt.figure(figsize=(10, 5))
plt.plot(hourly_total.index, hourly_total.values, label='Total Population', lw=2, color='#1f77b4')
plt.plot(hourly_dtou.index, hourly_dtou.values, label='dToU Dynamic Cohort', lw=2, linestyle='--', color='#d62728')
plt.plot(hourly_std.index, hourly_std.values, label='Standard Tariff Cohort', lw=2, linestyle=':', color='#2ca02c')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Mean Demand (kWh / 30-min)')
plt.title('Average Diurnal Electricity Demand Profile')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()